# 01 — CWRU Bearing Dataset: download, load, and plot a known fault

Starter EDA notebook for **VoltSense** (Phase 0). Goal: get one known-fault CWRU file
onto disk, load it, and plot the raw vibration signal so we can see what the replay
producer (Phase 1) will stream into Kafka.

> **No data is committed to the repo** (`data/` is git-ignored). Follow the download
> steps below to fetch it yourself.

## 1. Download the data (manual — do this once)

The CWRU Bearing Data Center publishes the files as MATLAB `.mat` downloads:

**https://engineering.case.edu/bearingdatacenter/download-data-file**

Recommended starter files (12 kHz Drive-End data, motor load 0 HP / ~1797 RPM):

| Condition | CWRU file | Suggested local name |
|---|---|---|
| Normal baseline | `Normal_0.mat` (97) | `data/cwru/normal_0.mat` |
| Inner-race fault, 0.007" | `IR007_0.mat` (105) | `data/cwru/ir007_0.mat` |
| Outer-race fault, 0.007" (@6) | `OR007@6_0.mat` (130) | `data/cwru/or007_0.mat` |
| Ball fault, 0.007" | `B007_0.mat` (118) | `data/cwru/b007_0.mat` |

Steps:
1. On the download page, pick **12k Drive End Bearing Fault Data** (and the Normal set).
2. Download the `.mat` files above.
3. Place them under `data/cwru/` in the repo root (create the folder).

```bash
mkdir -p ../data/cwru
# then move the downloaded .mat files into ../data/cwru/
```

Each file holds accelerometer time series. Variable names follow `X<id>_DE_time`
(drive-end), `X<id>_FE_time` (fan-end), `X<id>_BA_time` (base), plus `X<id>_RPM`.
The 12 kHz drive-end sets are sampled at **12,000 Hz**.

In [ ]:
# Dependencies (run once in your venv):
#   pip install scipy numpy matplotlib
import numpy as np
from scipy.io import loadmat
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("../data/cwru")
SAMPLE_RATE_HZ = 12_000  # 12 kHz drive-end sets

## 2. Loader

CWRU `.mat` variable names embed a per-file id (e.g. `X105_DE_time`), so we don't
hard-code them — we find the key by suffix.

In [ ]:
def load_cwru(path, channel="DE"):
    """Load a CWRU .mat file and return (signal, rpm).

    channel: 'DE' (drive-end), 'FE' (fan-end), or 'BA' (base) accelerometer.
    """
    mat = loadmat(path)
    sig_key = next((k for k in mat if k.endswith(f"_{channel}_time")), None)
    if sig_key is None:
        avail = [k for k in mat if not k.startswith("__")]
        raise KeyError(f"No '{channel}' channel in {path}. Available: {avail}")
    signal = mat[sig_key].ravel().astype(np.float64)

    rpm_key = next((k for k in mat if k.endswith("_RPM")), None)
    rpm = float(mat[rpm_key].ravel()[0]) if rpm_key else None
    return signal, rpm


# --- load one known-fault file (inner-race 0.007") ---
fault_file = DATA_DIR / "ir007_0.mat"   # adjust to whatever you downloaded
signal, rpm = load_cwru(fault_file, channel="DE")
print(f"file:    {fault_file.name}")
print(f"samples: {signal.size:,}  (~{signal.size / SAMPLE_RATE_HZ:.2f} s @ {SAMPLE_RATE_HZ} Hz)")
print(f"rpm:     {rpm}")
print(f"stats:   mean={signal.mean():.4f}  rms={np.sqrt(np.mean(signal**2)):.4f}  "
      f"peak={np.abs(signal).max():.4f}")

## 3. Plot the raw vibration signal

A short time-domain window — enough to see the periodic impacts a bearing fault
produces. (Frequency-domain analysis — FFT, envelope spectrum, bearing fault
frequencies — comes with the Flink feature job in Phase 1; see
`docs/fault-signatures.md`.)

In [ ]:
t = np.arange(signal.size) / SAMPLE_RATE_HZ
window = t < 0.1  # first 100 ms

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t[window], signal[window], lw=0.8)
ax.set_title(f"CWRU raw vibration — {fault_file.name} (DE, first 100 ms)")
ax.set_xlabel("time (s)")
ax.set_ylabel("acceleration (g)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Next (Phase 1)

- Compute the bearing fault frequencies (BPFO/BPFI/BSF/FTF) for the SKF 6205 drive-end
  bearing at this RPM and confirm energy appears at the expected band for the labelled
  fault (validate the moat).
- Frame this signal into records and replay it into Kafka via `producer/` → the
  `raw-telemetry` topic.